# Stage 0 — Data factory: OASIS-1 volumes → QC'd slices + splits

Converts raw T1 volumes into axial + coronal 16-bit PNG slices, joins the
cross-sectional clinical CSV (CDR, age, sex, nWBV…), and writes the
**subject-level** train/val/test splits + GroupKFold fold ids that every later
stage must use. **~15–30 min on Kaggle CPU.**

Checks performed here: slice QC counts, split disjointness, class balance.
If anything looks off, STOP — everything downstream inherits these files.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.factory import run_factory

cfg = load_config("configs/data_factory.yaml")
if IN_KAGGLE:
    cfg["root"] = "/kaggle/input/oasis1"   # <-- your attached dataset id here
summary = run_factory(cfg)

In [ ]:
# ---- VERIFICATION (do not skip) ----
import json
from pathlib import Path
import pandas as pd
md = pd.read_csv("data/factory/metadata.csv")
sp = pd.read_csv("data/factory/splits.csv")
assert sp.subject_id.is_unique, "LEAK: subject appears in two splits"
assert not set(sp[sp.split=="train"].subject_id) & set(sp[sp.split=="test"].subject_id)
print("slices:", len(md), "| subjects:", md.subject_id.nunique(),
      "| views:", md.view.value_counts().to_dict())
print("subject counts per split:", sp.split.value_counts().to_dict())
print("class counts (subjects):",
      md.drop_duplicates("subject_id").cdr_class.value_counts().sort_index().to_dict())
log = json.load(open("data/factory/factory_log.json"))
print("skipped visits:", log["n_skipped"], "| runtime:", log["runtime_sec"], "s")

In [ ]:
# eyeball a few slices straight from the factory output
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for v, row in zip(("axial", "coronal"), axes):
    allf = sorted(Path("data/factory/slices", v).glob("*.png"))
    files = allf[:: max(1, len(allf)//4)][:4]
    for ax, f in zip(row, files):
        ax.imshow(np.asarray(Image.open(f)) / 65535.0, cmap="gray"); ax.axis("off")
        ax.set_title(f.stem.split("_")[-2] + " " + f.stem.split("_")[-1], fontsize=8)
fig.suptitle("factory QC samples — top: axial, bottom: coronal (radiological)")
plt.tight_layout(); plt.show()

### Before closing this notebook
Commit the notebook version. `data/factory/` lives in `/kaggle/working` and is
versioned with the notebook — later notebooks can read it from there, or you
can publish it as a private Kaggle dataset for faster re-attach.